# DQN Atari Pong: Hyperparameter Tuning Experiments

## Overview

This notebook trains a Deep Q-Network (DQN) agent to play Atari Pong using
Stable Baselines 3 and Gymnasium. We run 10 experiments, each with a different
hyperparameter combination, then save all results to denis_experiments.csv.

### What is DQN?
DQN combines Q-learning with a deep neural network. Instead of a Q-table, a neural
network estimates Q(s,a), the expected future reward for taking action a in state s.
Two key tricks make it stable:
- Replay Buffer: stores past experiences and samples them randomly to break correlations
- Target Network: a slowly-updated copy of the network to stabilize Q-value targets

### Hyperparameters Being Tuned
| Hyperparameter | What it controls |
|---|---|
| lr | How fast the network updates its weights |
| gamma | How much future rewards are valued vs immediate rewards |
| batch_size | Number of experiences sampled per training update |
| exploration_fraction | How quickly epsilon decays from 1.0 to its final value |
| exploration_final_eps | Minimum exploration rate (agent never goes fully greedy) |
| policy | CnnPolicy (spatial pixel reading) vs MlpPolicy (flat vector) |


## 1. Setup

In [28]:
!pip install torch -q
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — go to Runtime > Change runtime type")

GPU available: True
Device: Tesla T4


In [29]:
!pip install stable-baselines3[extra] gymnasium[atari] ale-py autorom opencv-python-headless --quiet
!AutoROM --accept-license --quiet
print("All packages installed and Atari ROMs downloaded.")

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.
All packages installed and Atari ROMs downloaded.


In [30]:
import gymnasium as gym
import ale_py
import warnings
import logging
logging.getLogger('ale_py').setLevel(logging.ERROR)

env = gym.make('PongNoFrameskip-v4')
obs, info = env.reset()
print("Environment loaded successfully.")
print(f"  Observation shape : {obs.shape}")
print(f"  Action space      : {env.action_space}")
print(f"  Number of actions : {env.action_space.n}")
env.close()

Environment loaded successfully.
  Observation shape : (210, 160, 3)
  Action space      : Discrete(6)
  Number of actions : 6


##2. Upload the Scripts

Upload the three Python files:
- train.py — Training script
- play.py — Playback and evaluation script
- env_utils.py — Shared environment utilities imported by both scripts


In [31]:
import os
import shutil

DATASET_PATH = '/kaggle/input/datasets/denismitali/dqn-pong-scripts'

for f in ['train.py', 'play.py', 'env_utils.py']:
    src = os.path.join(DATASET_PATH, f)
    if os.path.exists(src):
        shutil.copy(src, f'/kaggle/working/{f}')
        print(f"  [OK]  {f}")
    else:
        print(f"  [MISSING]  {f}")

os.chdir('/kaggle/working')
print("\nReady.")

  [OK]  train.py
  [OK]  play.py
  [OK]  env_utils.py

Ready.


In [32]:
import os

required = ['train.py', 'play.py', 'env_utils.py']
all_present = True

for f in required:
    status = "OK" if os.path.exists(f) else "MISSING"
    print(f"  [{status}]  {f}")
    if not os.path.exists(f):
        all_present = False

print()
print("Ready to train." if all_present else "Re-upload the missing files before continuing.")

  [OK]  train.py
  [OK]  play.py
  [OK]  env_utils.py

Ready to train.


In [33]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('ale_py').setLevel(logging.ERROR)

with open('env_utils.py', 'w') as f:
    f.write('''import subprocess
import sys
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, VecTransposeImage

ENV_ID = "PongNoFrameskip-v4"
N_ENVS = 1
N_STACK = 4
SEED = 0

def check_requirements():
    required = ["stable_baselines3", "gymnasium", "ale_py"]
    for pkg in required:
        try:
            __import__(pkg)
        except ImportError:
            print(f"WARNING: {pkg} not installed")

def make_env(seed=None, render_mode=None, transpose_image=True):
    if seed is None:
        seed = SEED
    env_kwargs = {}
    if render_mode is not None:
        env_kwargs["render_mode"] = render_mode
    env = make_atari_env(
        ENV_ID,
        n_envs=N_ENVS,
        seed=seed,
        env_kwargs=env_kwargs if env_kwargs else None
    )
    env = VecFrameStack(env, n_stack=N_STACK)
    return env
''')

print("env_utils.py rewritten successfully.")

import importlib.util
spec = importlib.util.spec_from_file_location("env_utils", "env_utils.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
print("env_utils.py imports correctly. Ready to train.")

env_utils.py rewritten successfully.
env_utils.py imports correctly. Ready to train.


##3. Run All 10 Experiments

All experiments run sequentially in the single cell below.

In [34]:
import subprocess
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTHONWARNINGS'] = 'ignore'

def run_experiment(cmd, label):
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")

    env = os.environ.copy()
    env['TF_CPP_MIN_LOG_LEVEL'] = '3'
    env['PYTHONWARNINGS'] = 'ignore'

    process = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env
    )

    skip_patterns = [
        'Unable to register', 'WARNING:', 'W0000', 'E0000',
        'absl::InitializeLog', 'computation_placer',
        'cuda_dnn', 'cuda_blas', 'cuda_fft',
        'Gym has been unmaintained', 'Please upgrade to Gymnasium',
        'gymnasium.farama.org', 'migration_guide',
        'import gymnasium as gym', 'Users of this version',
        'Arcade Learning Environment', 'Powered by Stella',
        'UserWarning', 'warnings.warn', 'not of the same type',
        'VecTransposeImage', 'VecFrameStack',
        'tensorflow', 'TF_', 'cpu_feature_guard',
        'AVX', 'rebuild TensorFlow', 'lost sys.stderr',
        'Exception ignored', 'unraisablehook',
        'object address', 'object refcount', 'object type',
        'stable_baselines3', 'site-packages',
    ]

    for line in process.stdout:
        line_stripped = line.rstrip()
        if any(p in line_stripped for p in skip_patterns):
            continue
        if line_stripped:
            print(line_stripped)

    process.wait()

run_experiment('python -W ignore train.py --experiment_name "Exp01_Baseline" --member_name "Denis" --policy CnnPolicy --lr 1e-4 --gamma 0.99 --batch_size 32 --exploration_initial_eps 1.0 --exploration_final_eps 0.05 --exploration_fraction 0.1 --total_timesteps 1000000', 'Exp 1 — Baseline')

run_experiment('python -W ignore train.py --experiment_name "Exp02_HighLR" --member_name "Denis" --policy CnnPolicy --lr 5e-4 --gamma 0.99 --batch_size 32 --exploration_initial_eps 1.0 --exploration_final_eps 0.05 --exploration_fraction 0.1 --total_timesteps 300000', 'Exp 2 — High LR')

run_experiment('python -W ignore train.py --experiment_name "Exp03_LowLR" --member_name "Denis" --policy CnnPolicy --lr 1e-5 --gamma 0.99 --batch_size 32 --exploration_initial_eps 1.0 --exploration_final_eps 0.05 --exploration_fraction 0.1 --total_timesteps 300000', 'Exp 3 — Low LR')

run_experiment('python -W ignore train.py --experiment_name "Exp04_LowGamma" --member_name "Denis" --policy CnnPolicy --lr 1e-4 --gamma 0.80 --batch_size 32 --exploration_initial_eps 1.0 --exploration_final_eps 0.05 --exploration_fraction 0.1 --total_timesteps 300000', 'Exp 4 — Low Gamma')

run_experiment('python -W ignore train.py --experiment_name "Exp05_HighGamma" --member_name "Denis" --policy CnnPolicy --lr 1e-4 --gamma 0.999 --batch_size 32 --exploration_initial_eps 1.0 --exploration_final_eps 0.05 --exploration_fraction 0.1 --total_timesteps 500000', 'Exp 5 — High Gamma')

run_experiment('python -W ignore train.py --experiment_name "Exp06_LargeBatch" --member_name "Denis" --policy CnnPolicy --lr 1e-4 --gamma 0.99 --batch_size 64 --exploration_initial_eps 1.0 --exploration_final_eps 0.05 --exploration_fraction 0.1 --total_timesteps 500000', 'Exp 6 — Large Batch')

run_experiment('python -W ignore train.py --experiment_name "Exp07_BestCombo" --member_name "Denis" --policy CnnPolicy --lr 2.5e-4 --gamma 0.99 --batch_size 64 --exploration_initial_eps 1.0 --exploration_final_eps 0.01 --exploration_fraction 0.1 --buffer_size 100000 --total_timesteps 1000000', 'Exp 7 — Best Combo')

run_experiment('python -W ignore train.py --experiment_name "Exp08_FastEps" --member_name "Denis" --policy CnnPolicy --lr 1e-4 --gamma 0.99 --batch_size 32 --exploration_initial_eps 1.0 --exploration_final_eps 0.05 --exploration_fraction 0.05 --total_timesteps 300000', 'Exp 8 — Fast Epsilon Decay')

run_experiment('python -W ignore train.py --experiment_name "Exp09_SlowEps" --member_name "Denis" --policy CnnPolicy --lr 1e-4 --gamma 0.99 --batch_size 32 --exploration_initial_eps 1.0 --exploration_final_eps 0.05 --exploration_fraction 0.50 --total_timesteps 300000', 'Exp 9 — Slow Epsilon Decay')

run_experiment('python -W ignore train.py --experiment_name "Exp10_MLP" --member_name "Denis" --policy MlpPolicy --lr 1e-4 --gamma 0.99 --batch_size 32 --exploration_initial_eps 1.0 --exploration_final_eps 0.05 --exploration_fraction 0.1 --total_timesteps 300000', 'Exp 10 — MLP Policy')

print("\nAll experiments complete.")


  Exp 1 — Baseline
Using cuda device
Training with experiment: Exp01_Baseline
Member: Denis
Policy: CnnPolicy
Total timesteps: 1000000
Logging to runs/Exp01_Baseline_3
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 3.66e+03 |
|    ep_rew_mean      | -20.2    |
|    exploration_rate | 0.965    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 339      |
|    time_elapsed     | 10       |
|    total_timesteps  | 3636     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.000407 |
|    n_updates        | 883      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 3.65e+03 |
|    ep_rew_mean      | -20.1    |
|    exploration_rate | 0.931    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 348      |
|    time_elapsed     | 20

## 4. Results

All experiments append their results to results/experiments.csv automatically.
The cell below copies that file to denis_experiments.csv, loads it, sorts by
mean reward, and prints the final ranked table.

In [35]:
import os
import shutil
import pandas as pd

src = 'results/experiments.csv'
dst = 'results/denis_experiments.csv'

if os.path.exists(src):
    shutil.copy(src, dst)
    print(f"Results saved to {dst}")
else:
    print("No results file found — make sure at least one experiment completed successfully.")

df = pd.read_csv(dst)
df_sorted = df.sort_values('mean_reward', ascending=False).reset_index(drop=True)

cols = [
    'experiment_name', 'policy', 'lr', 'gamma', 'batch_size',
    'exploration_initial_eps', 'exploration_final_eps',
    'exploration_fraction', 'mean_reward', 'std_reward'
]

print()
print("=" * 80)
print("ALL EXPERIMENTS — Ranked best to worst")
print("=" * 80)
print(df_sorted[cols].to_string(index=False))
print()
print(f"Best  : {df_sorted.iloc[0]['experiment_name']}  ->  Mean Reward: {df_sorted.iloc[0]['mean_reward']:.2f}")
print(f"Worst : {df_sorted.iloc[-1]['experiment_name']}  ->  Mean Reward: {df_sorted.iloc[-1]['mean_reward']:.2f}")

Results saved to results/denis_experiments.csv

ALL EXPERIMENTS — Ranked best to worst
 experiment_name    policy      lr  gamma  batch_size  exploration_initial_eps  exploration_final_eps  exploration_fraction  mean_reward  std_reward
 Exp05_HighGamma CnnPolicy 0.00010  0.999          32                      1.0                   0.05                  0.10        -13.7    7.836453
Exp06_LargeBatch CnnPolicy 0.00010  0.990          64                      1.0                   0.05                  0.10        -14.5    1.431782
   Exp09_SlowEps CnnPolicy 0.00010  0.990          32                      1.0                   0.05                  0.50        -14.7    3.796051
     Exp03_LowLR CnnPolicy 0.00001  0.990          32                      1.0                   0.05                  0.10        -20.0    1.000000
   Exp08_FastEps CnnPolicy 0.00010  0.990          32                      1.0                   0.05                  0.05        -20.8    0.400000
  Exp01_Baseline Cn

In [39]:
import shutil
import os

src = 'results/experiments.csv'
dst = '/kaggle/working/denis_experiments.csv'

shutil.copy(src, dst)
print(f"File saved to: {dst}")

File saved to: /kaggle/working/denis_experiments.csv
